# Regional Sea Ice Timeseries

This notebook contains:
- Labrador Sea Timeseries
- Weddell Sea Timeseries
- Ross Sea Timeseries

In [ ]:
import calendar
import glob
import os

import xarray as xr
import statistics
import numpy as np
import yaml
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.path as mpath
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import nc_time_axis

import cupid_utils as cu

xr.set_options(file_cache_maxsize=1024);

In [ ]:
CESM_output_dir = (
    []
)  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"
ts_dir = None  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CESM_output_for_testing"

case_names = []
case_nicknames = []
start_dates = []
end_dates = []
case_align_years = []

obs_data_dir = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_obs_data"
path_model = ""  # "/glade/campaign/cesm/development/cross-wg/diagnostic_framework/CUPiD_model_data/ice/"
grid_file = ""  # "/glade/campaign/cesm/community/omwg/grids/tx2_3v2_grid.nc"
climo_nyears = 35

serial = False  # use dask LocalCluster

lc_kwargs = {}

In [ ]:
# Want some base case parameter defaults to equal control case values
if ts_dir is None:
    ts_dir = CESM_output_dir

In [ ]:
# When running interactively, cupid_run should be set to 0 for
# a DASK cluster

cupid_run = 1

if cupid_run == 1:

    from dask.distributed import Client, LocalCluster

    # Spin up cluster (if running in parallel)
    client = None
    if not serial:
        cluster = LocalCluster(**lc_kwargs)
        client = Client(cluster)

else:

    from dask.distributed import Client
    from dask_jobqueue import PBSCluster

    cluster = PBSCluster(
        cores=16,
        processes=16,
        memory="100GB",
        account="P93300065",
        queue="casper",
        walltime="02:00:00",
    )

    client = Client(cluster)

    cluster.scale(1)

    print(cluster)

client

In [ ]:
# Read in two cases. The ADF timeseries are needed here.

ds_grid = xr.open_dataset(grid_file)
if "nx" in ds_grid.dims:
    ds_grid = ds_grid.rename({"nx": "ni", "ny": "nj"})

if "TLAT" in ds_grid:
    TLAT = ds_grid["TLAT"]
    TLON = ds_grid["TLONG"]
    tarea = ds_grid["TAREA"] * 1.0e-4
    angle = ds_grid["ANGLE"]
else:
    TLAT = ds_grid["tlat"]
    TLON = ds_grid["tlon"]
    tarea = ds_grid["tarea"]
    angle = ds_grid["angle"]

path_lens1 = os.path.join(path_model, "cesm_lens1")
path_lens2 = os.path.join(path_model, "cesm_lens2")

path_cdr = os.path.join(
    obs_data_dir, "ice", "analysis_datasets", "hemispheric_data", "CDR_area_timeseries/"
)

In [ ]:
## set up dictionary for all cases listed

cmap = plt.get_cmap("tab10")
colors = cmap.colors[: len(case_names)]

cice_hist_vars = ["aice", "hi", "hs"]

case_info = {}
for case_name, case_nickname, start_date, end_date, case_align_year, tsdir in zip(
    case_names, case_nicknames, start_dates, end_dates, case_align_years, ts_dir
):

    # Only open the variables used here; opening every tseries file can exceed
    # xarray's file cache and cause "NetCDF: Not a valid ID" errors.
    ts_files = sorted(
        f
        for var in cice_hist_vars
        for f in glob.glob(
            os.path.join(
                tsdir,
                case_name,
                "ice",
                "proc",
                "tseries",
                f"{case_name}.cice.h.{var}.*.nc",
            )
        )
    )
    ds = xr.open_mfdataset(
        ts_files,
        data_vars="minimal",
        compat="override",
        coords="minimal",
    ).sel(time=slice(start_date, end_date))

    ds_ann = ds.resample(time="YS").mean(dim="time")
    climo_n_years = min(
        climo_nyears, len(ds_ann.time)
    )  ## had to name this something different than global climo_nyears set in params

    first_year = int(start_date.split("-")[0])
    end_year = int(end_date.split("-")[0])

    case_info[case_name] = {
        "case_nickname": case_nickname,
        "start_date": start_date,
        "end_date": end_date,
        "first_year": first_year,
        "end_year": end_year,
        "case_align_year": case_align_year,
        "ds": ds,
        "ds_ann": ds_ann,
        "climo_nyears": climo_n_years,
        "color": tuple(colors[case_names.index(case_name)]),
    }

In [ ]:
# Two functions to help draw plots even if only one year of data is available
# If one year is present, horizontal line will be dotted instead of solid


def da_plot_len_time_might_be_one(da_in, alt_time, color):
    # If da_in.time only has 1 value, draw horizontal line across range of alt_time
    if len(da_in.time) > 1:
        da_in.plot(color=color)
    else:
        time_arr = [alt_time.data[0], alt_time.data[-1]]
        plt.plot(time_arr, [da_in.data[0], da_in.data[0]], linestyle=":", color=color)


def plt_plot_len_x_might_be_one(da_in, x_in, alt_x, color):
    # If x_in only has one value, draw horizontal line across range of alt_x
    if len(x_in) > 1:
        plt.plot(x_in, da_in, color=color)
    else:
        plt.plot(
            [alt_x[0], alt_x[-1]],
            [da_in.data[0], da_in.data[0]],
            linestyle=":",
            color=color,
        )

In [ ]:
### Regional ice area, ice volume and snow volume from CESM1 LENS and CESM2 LENS
### Region masks follow /glade/work/dbailey/NOAA_NSIDC_CDR/compute_seaicearea_CDR.py

import glob
from matplotlib.path import Path

yml_dir = os.path.join(cu.__path__[0], "..", "input_files", "ice")
with open(os.path.join(yml_dir, "cice_masks.yml"), "r") as f:
    regions = yaml.safe_load(f)


def box_mask(lat, lon, lat_range, lon_range):
    """Mask for a lat/lon box. A lon range with lon_range[0] > lon_range[1] wraps through 0E."""
    m_lat = np.logical_and(lat > lat_range[0], lat < lat_range[1])
    if lon_range[0] > lon_range[1]:
        m_lon = np.logical_or(lon > lon_range[0], lon < lon_range[1])
    else:
        m_lon = np.logical_and(lon >= lon_range[0], lon < lon_range[1])
    return np.logical_and(m_lat, m_lon)


def poly_mask(lat, lon, poly_lat, poly_lon):
    """Mask for points inside a lat/lon polygon (lon in 0-360)."""
    path = Path(np.column_stack([poly_lon, poly_lat]))
    pts = np.column_stack([lon.ravel(), lat.ravel()])
    return path.contains_points(pts).reshape(lat.shape)


def get_range(key):
    # Some entries in the yml point to another entry by name (e.g. ArcOc_lat2: ArcOc_lat1)
    val = regions[key]
    return regions[val] if isinstance(val, str) else val


# Region definitions: name -> (hemisphere, kind, list of boxes or a polygon)
region_defs = {
    # Hemispheric totals
    "nh": ("nh", "hemi", None),
    "sh": ("sh", "hemi", None),
    # Northern Hemisphere boxes
    "lab": ("nh", "box", [("Lab_lat", "Lab_lon")]),
    "gin": ("nh", "box", [("GIN_lat", "GIN_lon1"), ("GIN_lat", "GIN_lon2")]),
    "bar": ("nh", "box", [("Bar_lat", "Bar_long")]),
    "arcoc": (
        "nh",
        "box",
        [
            ("ArcOc_lat1", "ArcOc_long1"),
            ("ArcOc_lat2", "ArcOc_long2"),
            ("ArcOc_lat3", "ArcOc_long3"),
        ],
    ),
    "sib": ("nh", "box", [("Sib_lat", "Sib_long")]),
    "beau": ("nh", "box", [("Beau_lat", "Beau_long")]),
    "carc": ("nh", "box", [("CArc_lat", "CArc_long")]),
    # Northern Hemisphere polygons
    "lab_poly": ("nh", "poly", ("Lab_poly_lat", "Lab_poly_long")),
    "bering": ("nh", "poly", ("Bering_poly_lat", "Bering_poly_long")),
    "okhotsk": ("nh", "poly", ("Okhotsk_poly_lat", "Okhotsk_poly_long")),
    "hudson": ("nh", "poly", ("Hudson_poly_lat", "Hudson_poly_long")),
    "caarch": ("nh", "poly", ("CAArch_poly_lat", "CAArch_poly_long")),
    # Southern Hemisphere boxes
    "wed": ("sh", "box", [("Wed_lat", "Wed_long1"), ("Wed_lat", "Wed_long2")]),
    "ross": ("sh", "box", [("Ross_lat", "Ross_long")]),
    "ind": ("sh", "box", [("Ind_lat", "Ind_long")]),
    "pac": ("sh", "box", [("Pac_lat", "Pac_long")]),
    "bam": ("sh", "box", [("BAm_lat", "BAm_long")]),
}


def region_weights(ds):
    """tarea masked to each region, as a (region, nj, ni) DataArray. Works on global or nh/sh-only grids."""
    lat = ds["TLAT"].values
    lon = ds["TLON"].values
    lon = np.where(lon < 0.0, lon + 360.0, lon)
    tarea = np.nan_to_num(ds["tarea"].values)
    weights = []
    for name, (hem, kind, spec) in region_defs.items():
        if kind == "hemi":
            mask = lat > 0.0 if hem == "nh" else lat < 0.0
        elif kind == "box":
            mask = np.zeros(lat.shape, dtype=bool)
            for lat_key, lon_key in spec:
                mask |= box_mask(lat, lon, get_range(lat_key), get_range(lon_key))
        else:
            mask = poly_mask(lat, lon, get_range(spec[0]), get_range(spec[1]))
        weights.append(np.where(mask, tarea, 0.0))
    return xr.DataArray(
        np.stack(weights),
        dims=["region", "nj", "ni"],
        coords={"region": list(region_defs)},
    )


# Output name for each input variable. Area in 10^6 km^2, volumes in 10^3 km^3.
out_names = {"aice": "area", "hi": "vhi", "hs": "vhs"}
out_units = {"aice": "1.0e6 km2", "hi": "1.0e3 km3", "hs": "1.0e3 km3"}


def regional_timeseries(files, var, weights):
    """Monthly regional sums of tarea*var over a list of files (one member, one grid)."""
    ds = xr.open_mfdataset(
        sorted(files),
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=True,
        chunks={"time": 120},
    )
    fld = ds[var]
    scale = 1.0e-12  # m^2 -> 10^6 km^2 and m^3 -> 10^3 km^3
    if fld.attrs.get("units") == "%":  # CESM1 aice is in percent, CESM2 is a fraction
        scale *= 0.01
    ts = xr.dot(fld.fillna(0.0), weights, dim=["nj", "ni"]) * scale
    ts["time"] = ds.time_bounds.mean(dim="d2")
    return ts


def to_dataset(da_ens):
    """(var, region, ensemble, time) -> Dataset with one variable per var/region, like the CDR file."""
    ds_out = xr.Dataset()
    for var in out_names:
        for name in region_defs:
            vname = out_names[var] + "_" + name
            ds_out[vname] = da_ens.sel(var=var, region=name, drop=True)
            ds_out[vname].attrs["units"] = out_units[var]
    return ds_out

In [ ]:
### Regional sea ice area for each case, using the region_weights masks

ds_weights = xr.Dataset({"TLAT": TLAT, "TLON": TLON, "tarea": tarea})
weights = region_weights(ds_weights)

for case_name, case_data in case_info.items():
    aice = case_data["ds"]["aice"]
    scale = 1.0e-12  # m^2 -> 10^6 km^2
    if aice.attrs.get("units") == "%":
        scale *= 0.01
    area = xr.dot(aice.fillna(0.0), weights, dim=["nj", "ni"]) * scale
    case_info[case_name]["area_regions"] = area.compute()

In [ ]:
### Read in the CESM LENS historical data

ds_cesm1_regional = xr.open_dataset(path_lens1 + "/cesm1_lens_regional_monthly.nc")
ds_cesm2_regional = xr.open_dataset(path_lens2 + "/cesm2_lens_regional_monthly.nc")

# if first_year >= 1850 or base_first_year >= 1850:
if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm1_years = np.linspace(1920, 2100, 181)
else:
    cesm1_years = np.linspace(1, 181, 181)

### Read in NOAA NASA CDR Timeseries

cdr_regions = xr.open_dataset(path_cdr + "CDR_sic_regional_monthly.nc")

# if first_year >= 1850 or base_first_year >= 1850:
if any(case["first_year"] >= 1850 for case in case_info.values()):
    cesm1_climo_start = 60
    cesm1_climo_end = 95
    cesm2_climo_start = 110
    cesm2_climo_end = 145
    cdr_climo_start = 0
    cdr_climo_end = 44
    cesm1_label = "CESM1-LENS Min / Max (1970-2005)"
    cesm2_label = "CESM2-LENS Min / Max (1979-2014)"
    CDR_label = "NOAA NASA CDR Min / Max (1979-2023)"
else:
    cesm1_climo_start = 0
    cesm1_climo_end = 35
    cesm2_climo_start = 0
    cesm2_climo_end = 35
    cdr_climo_start = 0
    cdr_climo_end = 21
    cesm1_label = "CESM1-LENS Min / Max (1920-1965)"
    cesm2_label = "CESM2-LENS Min / Max (1850-1885)"
    CDR_label = "NOAA NASA CDR Min / Max (1979-2000)"

In [ ]:
# # Set up legends

p1 = mlines.Line2D([], [], color="grey", linestyle="--", label=cesm1_label)
p2 = mlines.Line2D([], [], color="cyan", linestyle="--", label=cesm2_label)
p3 = [
    mlines.Line2D([], [], color=case_data["color"], label=case_data["case_nickname"])
    for case_name, case_data in case_info.items()
]
p4 = mlines.Line2D([], [], color="black", linestyle="-", label=CDR_label)

# Regional Timeseries Plots for Maximum Month

In [ ]:
# Circular boundary for the polar stereographic region maps
theta = np.linspace(0, 2 * np.pi, 100)
circle = mpath.Path(np.vstack([np.sin(theta), np.cos(theta)]).T * 0.5 + [0.5, 0.5])


def plot_region_map(ax, region):
    """Shade the grid cells in the region mask on a polar stereographic map."""
    if region_defs[region][0] == "nh":
        ax.set_extent([0.005, 360, 90, 45], crs=ccrs.PlateCarree())
    else:
        ax.set_extent([0.005, 360, -90, -45], crs=ccrs.PlateCarree())
    ax.set_boundary(circle, transform=ax.transAxes)
    ax.add_feature(cfeature.LAND, zorder=100, edgecolor="k")
    ax.gridlines(linestyle=":", color="grey")

    mask = np.where(weights.sel(region=region).values > 0.0, 1.0, np.nan)
    ax.pcolormesh(
        TLON.values,
        TLAT.values,
        mask,
        cmap="Reds",
        vmin=0.0,
        vmax=1.5,
        transform=ccrs.PlateCarree(),
    )


def plot_region(region, month, title):
    """Plot the regional sea ice area for one month from each case along with the CDR timeseries."""
    month_name = calendar.month_name[month]
    fig = plt.figure(figsize=(14, 5))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[3, 1], height_ratios=[3, 2])

    # Time series on the left, region map and legend stacked on the right
    fig.add_subplot(gs[:, 0])

    x1 = None
    x_end = 0
    for case_name, case_data in case_info.items():
        da = case_data["area_regions"].sel(region=region)
        da = da.sel(time=(da.time.dt.month == month))
        x = da.time.dt.year.values + case_data["case_align_year"]
        if x1 is None:
            x1 = x  # baseline x
        x_end = max(x_end, x[-1])
        plt_plot_len_x_might_be_one(da, x, x1, color=case_data["color"])

    cdr = cdr_regions["sic_monthly_" + region]
    cdr = cdr.sel(time=(cdr.time.dt.month == month))
    cdr = cdr.isel(time=slice(cdr_climo_start, cdr_climo_end))
    plt.axhline(float(cdr.max()), color="black", linestyle="-")
    plt.axhline(float(cdr.min()), color="black", linestyle="-")

    # CESM1-LENS range: climatological mean for the chosen month of each member over the
    # climo period, then the max and min across the ensemble
    cesm1_month = ds_cesm1_regional["area_" + region]
    cesm1_month = cesm1_month.sel(time=(cesm1_month.time.dt.month == month))
    cesm1_climo = cesm1_month.isel(time=slice(cesm1_climo_start, cesm1_climo_end)).mean(
        dim="time"
    )
    plt.axhline(float(cesm1_climo.max(dim="ensemble")), color="grey", linestyle="--")
    plt.axhline(float(cesm1_climo.min(dim="ensemble")), color="grey", linestyle="--")

    # CESM2-LENS range: climatological mean for the chosen month of each member over the
    # climo period, then the max and min across the ensemble
    cesm2_month = ds_cesm2_regional["area_" + region]
    cesm2_month = cesm2_month.sel(time=(cesm2_month.time.dt.month == month))
    cesm2_climo = cesm2_month.isel(time=slice(cesm2_climo_start, cesm2_climo_end)).mean(
        dim="time"
    )
    plt.axhline(float(cesm2_climo.max(dim="ensemble")), color="cyan", linestyle="--")
    plt.axhline(float(cesm2_climo.min(dim="ensemble")), color="cyan", linestyle="--")

    plt.grid(axis="y")
    plt.title(f"{title} {month_name} Mean Sea Ice Area")
    plt.xlabel("Year")
    plt.ylabel(f"{title} Ice Area $10^{{6}}$ km$^{{2}}$")

    proj = (
        ccrs.NorthPolarStereo()
        if region_defs[region][0] == "nh"
        else ccrs.SouthPolarStereo()
    )
    ax_map = fig.add_subplot(gs[0, 1], projection=proj)
    plot_region_map(ax_map, region)
    ax_map.set_title(f"{title} Region Mask")

    ax_leg = fig.add_subplot(gs[1, 1])
    ax_leg.axis("off")
    ax_leg.legend(handles=[p1, p2, p4, *p3], loc="center")

# Northern Hemisphere Regions

In [ ]:
plot_region("lab", 3, "Labrador Sea")
plot_region("gin", 3, "GIN Seas")
plot_region("bar", 3, "Barents Sea")
plot_region("arcoc", 3, "Arctic Ocean")
plot_region("sib", 3, "Siberian Seas")
plot_region("beau", 3, "Beaufort Sea")
plot_region("carc", 3, "Central Arctic Ocean")
plot_region("bering", 3, "Bering Sea")
plot_region("okhotsk", 3, "Sea of Okhotsk")
plot_region("hudson", 3, "Hudson Bay")
plot_region("caarch", 3, "Canadian Arctic Archipelago")

# Southern Hemisphere Regions

In [ ]:
plot_region("wed", 9, "Weddell Sea")
plot_region("ind", 9, "Indian Ocean Sector")
plot_region("pac", 9, "Pacific Sector")
plot_region("ross", 9, "Ross Sea")
plot_region("bam", 9, "Bellingshausen-Amundsen Seas")

In [ ]:
if client and not cupid_run:
    client.shutdown()